# 04 · Optimal Study Time — Time-Series Analysis

**Goal**: Identify each student's peak focus hours and best weekdays using activity timestamps,
weighted by quiz performance when available.

**Outputs**: `analytics/ml/study_time.py` — `compute_pattern()` + `get_recommendation()`

---

## 1 · Problem Framing

Research on *spaced practice* and *time-of-day effects* (Keskin & Yildirim 2018;
Folkard & Monk 1985) shows that memory consolidation peaks during *the student's own*
alertness window — not a universal clock.  Our goal is to **learn that window from
behavioural data** (chat, flashcard reviews, quiz attempts) rather than hard-coding it.

Key questions:
- Which **hour of day** does the user most consistently engage with the platform?
- Which **weekdays** show the highest study intensity?
- Can we weight activity by **quiz performance** to surface *productive* hours,
  not just *online* hours?
- How **confident** are we?  (n_events / 50 heuristic)

## 2 · Data Extraction

We pull from three Django models:
- `ChatHistory.timestamp`
- `FlashcardReview.reviewed_at`
- `QuizAttempt.attempted_at` + `score`

In [ ]:
import sys, os
# Add project root to path so Django ORM is available
sys.path.insert(0, os.path.abspath('..'))
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'lumen_project.settings')

import django
django.setup()

from django.contrib.auth.models import User
from core.models import ChatHistory, FlashcardReview, QuizAttempt

print('Users          :', User.objects.count())
print('ChatHistory    :', ChatHistory.objects.count())
print('FlashcardReview:', FlashcardReview.objects.count())
print('QuizAttempt    :', QuizAttempt.objects.count())

## 3 · EDA — Synthetic Data

Generate synthetic users to prototype the pipeline before real data accumulates.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from datetime import datetime, timedelta
import zoneinfo

rng = np.random.default_rng(42)
TZ  = zoneinfo.ZoneInfo('Asia/Kolkata')

def synthetic_user(n=200, peak_hour=20, peak_days=(0,1,4)):
    """Return a list of IST-aware datetimes with a realistic peak."""
    base = datetime(2024, 1, 1, tzinfo=TZ)
    records = []
    for _ in range(n):
        day_offset = rng.integers(0, 90)
        weekday    = (base + timedelta(days=int(day_offset))).weekday()
        # Hours biased toward peak
        if weekday in peak_days:
            hour = int(rng.normal(peak_hour, 1.2)) % 24
        else:
            hour = int(rng.integers(8, 23))
        ts = base + timedelta(days=int(day_offset), hours=hour,
                              minutes=int(rng.integers(0, 59)))
        records.append(ts.replace(tzinfo=TZ))
    return records

users_ts = [synthetic_user(200, peak_hour=h, peak_days=pd_)
            for h, pd_ in [(20,(0,1,4)),(9,(2,3)),(14,(5,6)),(22,(0,2,4))]]

# Hour histogram (averaged over all users)
all_hours = [dt.hour for ts in users_ts for dt in ts]
fig, ax = plt.subplots(figsize=(12, 3))
ax.bar(range(24), np.bincount(all_hours, minlength=24), color='#6366f1', alpha=.8)
ax.set_xlabel('Hour of day (local)')
ax.set_ylabel('Events')
ax.set_title('Overall activity by hour of day')
ax.set_xticks(range(24))
plt.tight_layout()
plt.show()

In [ ]:
# Weekday heatmap (all synthetic users)
DAY_NAMES = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
matrix_all = np.zeros((7,24))
for ts_list in users_ts:
    for dt in ts_list:
        matrix_all[dt.weekday(), dt.hour] += 1

fig, ax = plt.subplots(figsize=(14, 3))
im = ax.imshow(matrix_all, aspect='auto', cmap='Purples', interpolation='nearest')
ax.set_yticks(range(7)); ax.set_yticklabels(DAY_NAMES)
ax.set_xticks(range(24)); ax.set_xticklabels(range(24), fontsize=7)
ax.set_xlabel('Hour of day'); ax.set_title('Weekday × Hour activity heatmap (all users)')
plt.colorbar(im, ax=ax, label='Event count')
plt.tight_layout(); plt.show()

## 4 · Timezone Handling

All timestamps are stored as UTC in Django (`DateTimeField` with `USE_TZ=True`).
Converting to the user's local timezone is critical — an event at UTC 18:30 is
midnight IST (00:00 +05:30), which belongs to the *next calendar day*.

We use `zoneinfo.ZoneInfo` (Python 3.9+, stdlib) — no external dependency.
Default: `Asia/Kolkata`.  Future work: store per-user timezone preference.

In [ ]:
from datetime import timezone as dt_tz
utc_ts   = datetime(2024, 6, 15, 18, 30, 0, tzinfo=dt_tz.utc)
local_ts = utc_ts.astimezone(TZ)
print(f'UTC  : {utc_ts}')
print(f'IST  : {local_ts}   (hour={local_ts.hour}, day={local_ts.day})')
assert local_ts.hour == 0 and local_ts.day == 16, 'TZ conversion error!'
print('✓ Timezone conversion correct')

## 5 · Seasonal Decomposition

We flatten the 7×24 matrix to a 168-step series and run additive STL
decomposition with `period=24`.  The *trend* component smooths out noise
and is used for peak detection.

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

ts_list   = users_ts[0]  # first synthetic user
mat_u     = np.zeros((7,24))
for dt in ts_list:
    mat_u[dt.weekday(), dt.hour] += 1
mat_u_log = np.log1p(mat_u)
mat_u_log /= max(mat_u_log.max(), 1e-9)

series = pd.Series(mat_u_log.flatten())
result = seasonal_decompose(series, model='additive', period=24, extrapolate_trend='freq')

fig, axes = plt.subplots(4, 1, figsize=(14, 8), sharex=True)
for ax, comp, name in zip(axes,
    [series, result.trend, result.seasonal, result.resid],
    ['Observed','Trend','Seasonal','Residual']):
    ax.plot(comp, color='#6366f1', lw=1.2)
    ax.set_ylabel(name, fontsize=9)
    ax.grid(alpha=.2)
axes[-1].set_xlabel('Index (day*24 + hour)')
fig.suptitle('Seasonal Decomposition of Study Activity (User 0)', y=1.01)
plt.tight_layout(); plt.show()

## 6 · Productivity Weighting

When quiz score data is available, we blend:
```
weight = 0.6 * activity_norm  +  0.4 * avg_quiz_score_at_hour_norm
```
This elevates hours where the student not only studies *more* but also *performs better*.

In [ ]:
# Simulate quiz scores — better at peak hour
sim_scores = {h: float(rng.normal(70 + (10 if h == 20 else 0), 8)) for h in range(24)}
score_norm = np.array([max(0, sim_scores.get(h, 60)) / 100 for h in range(24)])

activity_norm = mat_u_log.mean(axis=0)   # 24-length
activity_norm /= max(activity_norm.max(), 1e-9)

combined = 0.6 * activity_norm + 0.4 * score_norm

fig, ax = plt.subplots(figsize=(12, 3))
x = range(24)
ax.bar(x, activity_norm, label='Activity (60%)', alpha=.7, color='#6366f1')
ax.bar(x, score_norm * 0.4, bottom=activity_norm * 0.6,
       label='Quiz score (40%)', alpha=.7, color='#f59e0b')
ax.plot(x, combined, 'r-o', ms=4, label='Combined weight', lw=1.5)
ax.set_xticks(x); ax.set_xlabel('Hour')
ax.set_title('Productivity-weighted activity by hour')
ax.legend()
plt.tight_layout(); plt.show()

## 7 · Peak-Window Detection

Algorithm: slide a window of length 2–3 hours over the 24-h averaged profile;
pick the start that maximises sum of weights.  Wrap-around is handled via modulo.

In [ ]:
from analytics.ml.study_time import _best_window, _build_matrix, _hourly_series, _seasonal_decompose_trend

for i, ts_list in enumerate(users_ts):
    mat  = _build_matrix(ts_list, {})
    ser  = _hourly_series(mat)
    trend = _seasonal_decompose_trend(ser)
    s, e = _best_window(trend)
    print(f'User {i}: peak window = {s:02d}h – {e:02d}h')

In [ ]:
# Per-user heatmap grid (4 users)
fig, axes = plt.subplots(2, 2, figsize=(14, 5))
for ax, ts_list, idx in zip(axes.flat, users_ts, range(4)):
    m = _build_matrix(ts_list, {})
    im = ax.imshow(m, aspect='auto', cmap='Purples', vmin=0, vmax=1)
    ax.set_title(f'User {idx}', fontsize=10)
    ax.set_yticks(range(7)); ax.set_yticklabels(DAY_NAMES, fontsize=7)
    ax.set_xticks(range(0,24,4)); ax.set_xticklabels(range(0,24,4), fontsize=7)
plt.suptitle('Per-user normalised study intensity (4 synthetic users)')
plt.tight_layout(); plt.show()

## 8 · Confidence vs Sample Size


In [ ]:
n_vals = range(0, 101)
conf   = [min(1.0, n / 50) for n in n_vals]

fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(n_vals, conf, color='#6366f1', lw=2)
ax.axvline(50, color='#f59e0b', ls='--', label='Full confidence (50 events)')
ax.set_xlabel('n_events'); ax.set_ylabel('confidence')
ax.set_title('Confidence score vs number of activity events')
ax.legend(); ax.grid(alpha=.2)
plt.tight_layout(); plt.show()

## 9 · Limitations

- **Sparse data**: Users with <10 events get a default recommendation; confidence shown to UI.
- **Timezone drift**: Default TZ is Asia/Kolkata. Per-user TZ preferences are not yet stored.
- **Weekend bias**: Students who only study on weekends can skew weekday detection.
- **Session length**: A long session counts as one event at start time — under-representing its duration.
- **Seasonal changes**: Study patterns shift with exam season; `computed_at` is refreshed every 24 h.

## 10 · Production Usage

```python
# Recompute one user
from analytics.ml.study_time import compute_pattern
compute_pattern(user_id=1)

# Nightly batch (run via cron or Celery)
# python manage.py compute_study_patterns --batch-size 100

# API response
from analytics.ml.study_time import get_recommendation
result = get_recommendation(user_id=1)
# → {peak_window, peak_days, heatmap[168], message, confidence, productivity_score}
```

See `analytics/ml/study_time.py` for full implementation.